In [ ]:
!pip install crewai -qq

In [ ]:
!pip install crewai_tools -qq

In [ ]:
import os
os.environ["MISTRAL_API_KEY"]="2smqxCpjSh1yZeAjZUs4yUafsIJbwmbl"

In [ ]:
from crewai import Agent, Task, Crew, Process
from crewai_tools import FileReadTool
import pandas as pd

# Create tool for file reading
read_csv_tool = FileReadTool(file_path='/content/Synthetic_Financial_datasets_log.csv')

# Agents
data_collector = Agent(
    role="Data Collector",
    goal="Load and profile the dataset using pandas, not by hallucinating.",
    backstory="You are responsible for loading and summarizing the real dataset from a CSV file.",
    tools=[read_csv_tool],   # <-- give ability to read files
    verbose=True,
    reasoning=True,
    memory=True
)

pattern_recognizer = Agent(
    role="Pattern Recognizer",
    goal="Detect suspicious transactions using the actual dataset.",
    backstory="You analyze high-value amounts, suspicious transaction types (TRANSFER, CASH_OUT), and balance inconsistencies.",
    tools=[read_csv_tool],
    verbose=True,
    reasoning=True,
    memory=True

)

reporter = Agent(
    role="Fraud Reporter",
    goal="Generate a fraud detection report summarizing anomalies.",
    backstory="You prepare professional reports based on the actual dataset findings.",
    verbose=True,
    reasoning=True,
    memory=True
)

# Tasks
load_task = Task(
    description=(
    "Use FileReadTool to analyze anomalies in batches of 500 rows at a time "
    "from '/content/Synthetic_Financial_datasets_log.csv'. "
    "Focus on suspicious transaction types (TRANSFER, CASH_OUT), very large amounts, "
    "and balance inconsistencies. Summarize anomalies with row indices and amounts."
    ),
    agent=data_collector,
    expected_output="Dataset profile with row count, column names, dtypes, missing values, and 5 real sample rows.",
    input_data={"dataset_path": "PS_20174392719_1491204439457_log.csv"}
)

detect_task = Task(
    description=(
        "Analyze the loaded dataset to identify anomalies: "
        "very high transaction amounts, suspicious types (TRANSFER, CASH_OUT), "
        "and balance inconsistencies. Provide examples with row indices."
    ),
    agent=pattern_recognizer,
    expected_output="A list of detected anomalies with explanations."
)

report_task = Task(
    description="Prepare a structured fraud detection report summarizing anomalies and recommendations.",
    agent=reporter,
    expected_output="Fraud detection report (executive summary + findings + recommendations)."
)

# Crew
crew = Crew(
    agents=[data_collector, pattern_recognizer, reporter],
    tasks=[load_task, detect_task, report_task],
    verbose=True,
    process=Process.sequential,  # The manager oversees the task flow
    planning=True
)

# Run
result = crew.kickoff()
print("\n=== Final Fraud Report ===\n")
print(result)
